# Sistema de Predição de Doença Cardíaca
### Relatório dos Experimentos — Etapas 1 e 2

**Problema:** classificação binária — prever se um paciente apresenta indício de
doença cardíaca (`target = 1`) ou não (`target = 0`) a partir de variáveis clínicas.

**Objetivo preditivo:** apoiar a triagem clínica estimando o risco de doença
cardíaca com base em exames e características do paciente.

**Base de dados:** UCI Heart Disease — *Cleveland* (303 registros, 13 variáveis
explicativas + 1 alvo). Fonte original: UCI Machine Learning Repository.

---

### Papéis e responsabilidades da equipe
*(preencher com os integrantes do grupo)*

| Integrante | Responsabilidade |
|------------|------------------|
| Fulano     | Coleta e pré-processamento |
| Ciclano    | Análise descritiva |
| Beltrano   | Experimentos e modelagem |
| ...        | API e frontend (Flask) |


## 1. Coleta dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Carregamento direto da base (espelho da UCI Heart Disease - Cleveland)
URL = "https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv"
df = pd.read_csv(URL)

print("Dimensoes:", df.shape)
df.head()

## 2. Identificação e categorização das variáveis

**Variável alvo (Y):** `target` (0 = sem doença, 1 = com doença).

**Variáveis explicativas (X):**

| Variável | Descrição | Tipo |
|----------|-----------|------|
| age | Idade em anos | Numérica |
| sex | Sexo (1 = masculino, 0 = feminino) | Categórica |
| cp | Tipo de dor no peito (0–3) | Categórica |
| trestbps | Pressão arterial em repouso (mmHg) | Numérica |
| chol | Colesterol sérico (mg/dl) | Numérica |
| fbs | Glicemia em jejum > 120 mg/dl (1/0) | Categórica |
| restecg | Resultado do ECG em repouso (0–2) | Categórica |
| thalach | Frequência cardíaca máxima atingida | Numérica |
| exang | Angina induzida por exercício (1/0) | Categórica |
| oldpeak | Depressão de ST induzida por exercício | Numérica |
| slope | Inclinação do segmento ST (0–2) | Categórica |
| ca | Nº de vasos principais coloridos (0–4) | Categórica |
| thal | Talassemia (0–3) | Categórica |


In [ ]:
numericas = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categoricas = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
alvo = "target"

print("Numéricas:", numericas)
print("Categóricas:", categoricas)
print("Alvo:", alvo)
df.info()

## 3. Pré-processamento dos dados

In [ ]:
# 3.1 Valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum())

In [ ]:
# 3.2 Inconsistências / duplicatas
print("Linhas duplicadas:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Após remoção:", df.shape)

In [ ]:
# 3.3 Verificação de balanceamento do alvo
contagem = df[alvo].value_counts()
print(contagem)
print("\nProporção:")
print((contagem / len(df) * 100).round(1))

**Observações do pré-processamento:**
- A base não possui valores ausentes.
- Havia 1 registro duplicado, que foi removido.
- O alvo está razoavelmente balanceado (~54% / 46%), portanto não foi necessário
  rebalanceamento.
- O escalonamento (`StandardScaler`) é aplicado **dentro do pipeline** de cada
  modelo, evitando vazamento de dados entre treino e teste.


## 4. Análise descritiva

In [ ]:
# 4.1 Estatísticas descritivas das variáveis numéricas
df[numericas].describe().round(2)

In [ ]:
# 4.2 Histogramas das variáveis numéricas
df[numericas].hist(figsize=(12, 7), bins=20, color="#457b9d", edgecolor="white")
plt.suptitle("Distribuição das variáveis numéricas", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Gráfico de barras: distribuição do alvo
plt.figure(figsize=(5, 4))
sns.countplot(x=alvo, data=df, palette=["#2a9d8f", "#e76f51"])
plt.title("Distribuição da variável alvo")
plt.xticks([0, 1], ["Sem doença", "Com doença"])
plt.xlabel("")
plt.show()

In [ ]:
# 4.4 Gráfico de dispersão: idade x frequência cardíaca máxima, por classe
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="age", y="thalach", hue=alvo,
                palette=["#2a9d8f", "#e76f51"])
plt.title("Idade vs. Frequência cardíaca máxima")
plt.legend(title="Doença", labels=["Sem", "Com"])
plt.show()

In [ ]:
# 4.5 Matriz de correlação
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            annot_kws={"size": 7})
plt.title("Matriz de correlação")
plt.show()

**Principais descobertas da análise descritiva:**
- `cp` (tipo de dor no peito), `thalach` (FC máxima) e `slope` têm correlação
  positiva relevante com a presença de doença.
- `exang`, `oldpeak` e `ca` têm correlação negativa relevante.
- A FC máxima tende a ser **maior** nos pacientes com diagnóstico positivo nesta base.


## 5. Experimentos e escolha do melhor modelo

Foram treinados **5 modelos**, cada um executado **30 vezes** com divisões
treino/teste diferentes (estratificadas), registrando média e desvio padrão das
métricas — o que permite avaliar não só o desempenho mas também a **estabilidade**
de cada modelo.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

features = numericas + categoricas
X = df[features].values
y = df[alvo].values

modelos = {
    "Regressão Logística": Pipeline([("sc", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=1000))]),
    "Árvore de Decisão":   Pipeline([("sc", StandardScaler()),
                                     ("clf", DecisionTreeClassifier())]),
    "Random Forest":       Pipeline([("sc", StandardScaler()),
                                     ("clf", RandomForestClassifier(n_estimators=200))]),
    "SVM":                 Pipeline([("sc", StandardScaler()),
                                     ("clf", SVC(probability=True))]),
    "KNN":                 Pipeline([("sc", StandardScaler()),
                                     ("clf", KNeighborsClassifier(n_neighbors=7))]),
}

N = 30
linhas = []
for nome, modelo in modelos.items():
    accs, precs, recs, f1s = [], [], [], []
    for i in range(N):
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.2, random_state=i, stratify=y)
        modelo.fit(Xtr, ytr)
        p = modelo.predict(Xte)
        accs.append(accuracy_score(yte, p))
        precs.append(precision_score(yte, p, zero_division=0))
        recs.append(recall_score(yte, p, zero_division=0))
        f1s.append(f1_score(yte, p, zero_division=0))
    linhas.append({
        "Modelo": nome,
        "Acurácia": f"{np.mean(accs):.3f} ± {np.std(accs):.3f}",
        "Precisão": f"{np.mean(precs):.3f} ± {np.std(precs):.3f}",
        "Recall":   f"{np.mean(recs):.3f} ± {np.std(recs):.3f}",
        "F1-Score": f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
        "_f1": np.mean(f1s),
    })

resultados = pd.DataFrame(linhas)
resultados.drop(columns="_f1").set_index("Modelo")

In [ ]:
# Visualização comparativa do F1-Score médio
ordenado = resultados.sort_values("_f1", ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(data=ordenado, y="Modelo", x="_f1", palette="viridis")
plt.xlabel("F1-Score médio (30 execuções)")
plt.title("Comparação dos modelos")
plt.xlim(0.7, 0.9)
plt.show()

## 6. Escolha do modelo final e justificativa

A **Regressão Logística** apresentou o maior F1-Score médio, com baixo desvio
padrão — ou seja, além de bom desempenho, é o modelo mais **estável** entre as
execuções. Soma-se a isso a sua simplicidade e interpretabilidade, vantajosas em
um contexto clínico. Por isso, foi escolhida como modelo final.


In [ ]:
melhor = resultados.sort_values("_f1", ascending=False).iloc[0]["Modelo"]
print("Modelo escolhido:", melhor)

# Treino final em 100% dos dados e exportação com joblib
import joblib
modelo_final = modelos[melhor]
modelo_final.fit(X, y)

pacote = {"pipeline": modelo_final, "features": features, "nome_modelo": melhor}
joblib.dump(pacote, "modelo_preditivo.pkl")
print("Modelo exportado em modelo_preditivo.pkl")

## 7. Considerações finais

- **Resultados:** os modelos lineares e baseados em distância (Regressão Logística,
  SVM, KNN) tiveram desempenho semelhante e superior à Árvore de Decisão isolada.
- **Limitações:** base pequena (302 registros) e de uma única instituição (Cleveland),
  o que limita a generalização.
- **Melhorias futuras:** combinar os datasets de Hungria/Suíça/Long Beach para
  ampliar a base, otimizar hiperparâmetros (GridSearch) e validar com validação
  cruzada k-fold.

O arquivo `modelo_preditivo.pkl` gerado aqui é o que será carregado pela aplicação
Flask na Etapa 3.
